# Stage 3 — Content extraction agent (fine-tuned Qwen on Hugging Face)

Reads the kept frames from Stage 2, runs your fine-tuned Qwen vision-language model on each one, and writes the extracted content into the manifest — one JSON record per frame, timestamps preserved.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine for 2B/3B models; 7B needs the 4-bit option or an A100).

Two ways to run the model — pick one in the config cell:
- **`local`** — download the model into the Colab GPU (default; free, needs enough VRAM)
- **`endpoint`** — call a Hugging Face Inference Endpoint you've deployed (no GPU runtime needed)

Progress is **checkpointed after every frame**, so a crash or disconnect resumes where it left off.

In [ ]:
# @title 1. Setup — install dependencies & check GPU { display-mode: "form" }
!pip install -q "transformers>=4.49" accelerate qwen-vl-utils bitsandbytes peft pillow huggingface_hub

import json, math, re, shutil, time
from dataclasses import dataclass, asdict
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB VRAM)")
else:
    print("⚠️ No GPU — Runtime → Change runtime type → GPU (or use RUN_MODE='endpoint')")


In [ ]:
# @title 2. Stage 3 configuration { display-mode: "form" }

@dataclass
class Stage3Config:
    # --- your model ---
    model_id: str = "shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm"
    is_lora_adapter: bool = True        # this repo is an Unsloth-trained LoRA adapter
    base_model_id: str = "Qwen/Qwen2.5-VL-7B-Instruct"
                                        # official base + on-the-fly 4-bit quantization (below).
                                        # The unsloth pre-quantized repo the adapter was trained from
                                        # ("unsloth/qwen2.5-vl-7b-instruct-unsloth-bnb-4bit") breaks on
                                        # current transformers — the LoRA attaches identically to this one.
    run_mode: str = "local"             # "local" (load into Colab GPU) or "endpoint" (HF Inference Endpoint)
    endpoint_url: str = ""              # only for run_mode="endpoint", e.g. "https://xxxx.endpoints.huggingface.cloud"

    # --- generation ---
    max_new_tokens: int = 1024
    temperature: float = 0.1            # near-deterministic: we want faithful transcription, not creativity
    load_in_4bit: bool = True           # quantize the official base on load → ~7 GB VRAM, fits a T4
    max_retries: int = 3

    # --- frames input ---
    frame_interval_sec: int = 30        # ONLY used to reconstruct timestamps when manifest.json is absent
    image_format: str = "jpg"

    # --- paths ---
    work_dir: str = "/content/stage3"

    def __post_init__(self):
        self.frames_dir    = str(Path(self.work_dir) / "frames_kept")
        self.manifest_path = str(Path(self.work_dir) / "manifest.json")

CFG = Stage3Config()
Path(CFG.work_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.frames_dir).mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(CFG), indent=2))


In [ ]:
# @title 3. Hugging Face login — OPTIONAL: your adapter repo is public (Apache-2.0),
# so you can SKIP this cell entirely. Kept here in case you later use private repos.
# Preferred: add your token as a Colab secret named HF_TOKEN (key icon in the left sidebar),
# with "Notebook access" enabled. Falls back to an interactive login prompt.
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("✅ Logged in via Colab secret HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret found — interactive login:")
    from huggingface_hub import notebook_login
    notebook_login()


## Input — the frames from Stage 2

Zip your folder of kept frames on your PC (include `manifest.json` in the zip if you have it — timestamps and dedup provenance carry over). Then use Option A to upload the zip, or Option B if it's already on Drive.

If there's **no manifest**, one is reconstructed from the filenames: `frame_0007.jpg` → timestamp `(7−1) × frame_interval_sec`. Set `frame_interval_sec` in the config cell to whatever you used in Stage 1 (30 in your case).

In [ ]:
# @title Option A — upload a zip of the frames folder
from google.colab import files
import zipfile

up = files.upload()
zip_name = next(iter(up.keys()))
with zipfile.ZipFile(zip_name) as z:
    z.extractall(CFG.work_dir + "/_incoming")
print("Extracted", zip_name)


In [ ]:
# @title Option B — frames already on Google Drive
from google.colab import drive
drive.mount("/content/drive")

INCOMING = "/content/drive/MyDrive/pipeline/frames_kept"   # ← EDIT: folder containing the images
shutil.copytree(INCOMING, CFG.work_dir + "/_incoming", dirs_exist_ok=True)
print("Copied from", INCOMING)


In [ ]:
# @title 4. Collect frames + load or reconstruct the manifest

incoming = Path(CFG.work_dir) / "_incoming"
exts = {".jpg", ".jpeg", ".png", ".webp"}

# gather images from anywhere inside the extracted zip / copied folder
images = sorted(p for p in incoming.rglob("*") if p.suffix.lower() in exts)
assert images, f"No images found under {incoming}"
for p in images:
    shutil.copy(p, Path(CFG.frames_dir) / p.name)
images = sorted(Path(CFG.frames_dir).glob("*"))

# try to find a manifest that came along
found_manifest = next(incoming.rglob("manifest.json"), None)

def frame_index(name: str) -> int:
    m = re.search(r"(\d+)", name)
    return int(m.group(1)) if m else 0

if found_manifest:
    manifest = json.loads(found_manifest.read_text())
    by_id = {f["frame_id"]: f for f in manifest["frames"]}
    frames = []
    for p in images:
        rec = by_id.get(p.stem, None)
        if rec is None:                       # image present but not in manifest — synthesize
            idx = frame_index(p.stem)
            ts = (idx - 1) * CFG.frame_interval_sec
            rec = {"frame_id": p.stem, "index": idx - 1, "timestamp_sec": ts,
                   "timestamp_hms": time.strftime("%H:%M:%S", time.gmtime(ts)),
                   "status": "kept"}
        rec["path"] = str(p)
        frames.append(rec)
    manifest["frames"] = frames
    print(f"✅ Loaded manifest ({len(frames)} frames matched to images)")
else:
    frames = []
    for p in images:
        idx = frame_index(p.stem)
        ts = max(0, (idx - 1)) * CFG.frame_interval_sec
        frames.append({"frame_id": p.stem, "index": idx - 1,
                       "timestamp_sec": ts,
                       "timestamp_hms": time.strftime("%H:%M:%S", time.gmtime(ts)),
                       "path": str(p), "status": "kept"})
    manifest = {"video": None,
                "created_utc": datetime.now(timezone.utc).isoformat(),
                "stages": {}, "frames": frames}
    print(f"⚠️ No manifest.json found — reconstructed one from {len(frames)} filenames "
          f"(timestamps assume 1 frame per {CFG.frame_interval_sec}s of the ORIGINAL video)")

Path(CFG.manifest_path).write_text(json.dumps(manifest, indent=2))
for f in manifest["frames"][:5]:
    print(f'{f["frame_id"]}  @ {f["timestamp_hms"]}')
print(f'… {len(manifest["frames"])} frames total')


## Load the model

Handles all four combinations: merged fine-tune or LoRA adapter × full precision or 4-bit. `AutoModelForImageTextToText` + `AutoProcessor` cover both Qwen2-VL and Qwen2.5-VL architectures.

Rough VRAM guide: 2B/3B ≈ 7–9 GB full precision (fits T4) · 7B ≈ 17 GB full / ~7 GB with `load_in_4bit=True`.

In [ ]:
# @title 5. Load model + processor (skip this cell if run_mode="endpoint")
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

model, processor = None, None
if CFG.run_mode == "local":
    # T4 GPUs don't support bfloat16 — pick compute dtype accordingly
    compute_dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)

    load_kwargs = {"device_map": "auto"}
    if "bnb-4bit" in CFG.base_model_id or "bnb-4bit" in CFG.model_id:
        # pre-quantized repo: pass NOTHING extra, let its baked-in quantization config drive.
        # NOTE: unsloth's dynamically-quantized repos break on some transformers versions —
        # if this path errors, use the official base + load_in_4bit=True instead (the default).
        print("Base repo is pre-quantized 4-bit — deferring to its baked-in config.")
    elif CFG.load_in_4bit:
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True)
        print(f"Quantizing to 4-bit on load (compute dtype: {compute_dtype})")
    else:
        load_kwargs["torch_dtype"] = "auto"

    load_id = CFG.base_model_id if CFG.is_lora_adapter else CFG.model_id
    t0 = time.time()
    model = AutoModelForImageTextToText.from_pretrained(load_id, **load_kwargs)

    if CFG.is_lora_adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, CFG.model_id)
        print("Attached LoRA adapter:", CFG.model_id)
        print("Active adapters:", model.active_adapters)

    try:
        processor = AutoProcessor.from_pretrained(CFG.model_id)
    except Exception:
        processor = AutoProcessor.from_pretrained(CFG.base_model_id)

    model.eval()
    print(f"✅ Model ready in {time.time()-t0:.0f}s")
else:
    print("run_mode='endpoint' — model stays on HF infrastructure, nothing to load here.")


## The content extraction agent — two-pass, fine-tune-friendly

Your adapter was trained for figure description, so we don't force it to emit JSON. Instead:

**Pass 1 (vision)** — a natural description prompt, shaped like the captioning task the adapter was tuned on: transcribe all text/code verbatim, describe any figure. The model does what it's good at.

**Pass 2 (text-only, same model, no image)** — a cheap structuring call that converts Pass 1's description into the JSON schema. Text-only structuring is a task base Qwen handles fine even through a captioning adapter, and it costs a fraction of the vision pass.

**Fallback (no model at all)** — if Pass 2 still doesn't yield valid JSON, a heuristic structurer extracts fenced code, guesses a title from the first line, and tags the frame type by keywords. The raw description is **always** stored in `raw_description` regardless, so nothing is ever lost and Stage 5's textbook comparison can use the full text.

Retries, per-frame checkpointing, and resume behavior are unchanged.

In [ ]:
# @title 6. Content extraction agent (describe → structure)

DESCRIBE_PROMPT = """Describe this frame from a technical video course in detail.
Transcribe every piece of visible text exactly as shown: the title, all bullet points, labels, and captions.
If code is visible, reproduce it verbatim, preserving formatting.
If there is a figure, chart, or diagram, describe what it depicts and how it is structured."""

STRUCTURE_PROMPT = """Convert the following description of a video frame into a JSON object with exactly these fields:
{{"slide_title": string or null, "content_text": string, "code": string or null, "diagram_description": string or null, "frame_type": "slide" | "code_editor" | "terminal" | "diagram" | "talking_head" | "other"}}
Rules: put transcribed body text in content_text; put verbatim code in code; put figure/diagram explanation in diagram_description; use null when absent.
Respond with ONLY the JSON object.

Description:
{description}"""


def _lenient_json(text: str):
    t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", t, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                pass
    return None


def _structure_heuristic(description: str) -> dict:
    """Model-free fallback: structure a raw description with simple rules."""
    text = description.strip()
    code = None
    fences = re.findall(r"```[a-zA-Z]*\n?(.*?)```", text, re.DOTALL)
    if fences:
        code = "\n\n".join(f.strip() for f in fences)
        text = re.sub(r"```[a-zA-Z]*\n?.*?```", "", text, flags=re.DOTALL).strip()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    title = lines[0] if lines and len(lines[0]) < 80 else None
    low = description.lower()
    if code or "code editor" in low or "ide" in low:
        ftype = "code_editor"
    elif any(k in low for k in ("terminal", "command line", "shell prompt")):
        ftype = "terminal"
    elif any(k in low for k in ("chart", "graph", "diagram", "figure", "plot", "axis")):
        ftype = "diagram"
    elif any(k in low for k in ("person", "presenter", "speaker", "face", "webcam")):
        ftype = "talking_head"
    else:
        ftype = "slide"
    return {"slide_title": title, "content_text": text, "code": code,
            "diagram_description": text if ftype == "diagram" else None,
            "frame_type": ftype, "_structured_by": "heuristic"}


class ContentExtractionAgent:
    def __init__(self, cfg, model=None, processor=None):
        self.cfg, self.model, self.processor = cfg, model, processor
        if cfg.run_mode == "endpoint":
            from huggingface_hub import InferenceClient
            self.client = InferenceClient(base_url=cfg.endpoint_url)

    # ---------- one frame: describe → structure ----------
    def extract(self, image_path: str) -> dict:
        last_err = None
        for attempt in range(1, self.cfg.max_retries + 1):
            try:
                description = self._describe(image_path)          # pass 1 (vision)
                structured = self._structure(description)          # pass 2 (text-only)
                if structured is None:
                    structured = _structure_heuristic(description) # fallback (no model)
                structured["raw_description"] = description        # never lose the original
                return structured
            except Exception as e:
                last_err = e
                wait = 2 ** attempt
                print(f"  retry {attempt}/{self.cfg.max_retries} after error: {e} (waiting {wait}s)")
                time.sleep(wait)
        return {"slide_title": None, "content_text": None, "code": None,
                "diagram_description": None, "frame_type": "error",
                "raw_description": None, "_error": str(last_err)}

    def _describe(self, image_path: str) -> str:
        content = [{"type": "image", "image": image_path},
                   {"type": "text", "text": DESCRIBE_PROMPT}]
        return self._chat(content, max_new_tokens=self.cfg.max_new_tokens)

    def _structure(self, description: str):
        prompt = STRUCTURE_PROMPT.format(description=description)
        raw = self._chat([{"type": "text", "text": prompt}], max_new_tokens=self.cfg.max_new_tokens)
        parsed = _lenient_json(raw)
        if parsed is not None and isinstance(parsed, dict):
            parsed.setdefault("frame_type", "other")
            parsed["_structured_by"] = "model"
            return parsed
        return None

    # ---------- shared chat call, image optional ----------
    def _chat(self, content: list, max_new_tokens: int) -> str:
        if self.cfg.run_mode == "endpoint":
            return self._chat_endpoint(content, max_new_tokens)
        from qwen_vl_utils import process_vision_info
        messages = [{"role": "user", "content": content}]
        text = self.processor.apply_chat_template(messages, tokenize=False,
                                                  add_generation_prompt=True)
        has_image = any(c.get("type") == "image" for c in content)
        image_inputs = process_vision_info(messages)[0] if has_image else None
        inputs = self.processor(text=[text], images=image_inputs,
                                padding=True, return_tensors="pt").to(self.model.device)
        with torch.inference_mode():
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens,
                                      do_sample=self.cfg.temperature > 0,
                                      temperature=max(self.cfg.temperature, 1e-5))
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
        return self.processor.batch_decode(trimmed, skip_special_tokens=True,
                                           clean_up_tokenization_spaces=False)[0]

    def _chat_endpoint(self, content: list, max_new_tokens: int) -> str:
        import base64
        parts = []
        for c in content:
            if c["type"] == "image":
                b64 = base64.b64encode(Path(c["image"]).read_bytes()).decode()
                parts.append({"type": "image_url",
                              "image_url": {"url": f"data:image/jpeg;base64,{b64}"}})
            else:
                parts.append({"type": "text", "text": c["text"]})
        resp = self.client.chat.completions.create(
            model="tgi", messages=[{"role": "user", "content": parts}],
            max_tokens=max_new_tokens, temperature=self.cfg.temperature)
        return resp.choices[0].message.content

    # ---------- whole manifest, checkpointed + resumable ----------
    def run(self, manifest: dict) -> dict:
        todo = [f for f in manifest["frames"]
                if f.get("status") in (None, "kept") and "extracted_content" not in f]
        done = len([f for f in manifest["frames"] if "extracted_content" in f])
        print(f"{len(todo)} frames to process ({done} already done — resuming)")

        t0 = time.time()
        for i, rec in enumerate(todo, 1):
            t = time.time()
            rec["extracted_content"] = self.extract(rec["path"])
            rec["extraction_meta"] = {"model": self.cfg.model_id,
                                      "mode": self.cfg.run_mode,
                                      "structured_by": rec["extracted_content"].get("_structured_by", "error"),
                                      "elapsed_sec": round(time.time() - t, 1)}
            Path(self.cfg.manifest_path).write_text(json.dumps(manifest, indent=2))
            title = rec["extracted_content"].get("slide_title") or "—"
            print(f'[{i}/{len(todo)}] {rec["frame_id"]} @ {rec["timestamp_hms"]} '
                  f'({rec["extraction_meta"]["elapsed_sec"]}s, '
                  f'{rec["extraction_meta"]["structured_by"]}) → {str(title)[:60]}')

        manifest.setdefault("stages", {})["content_extraction"] = {
            "model": self.cfg.model_id, "mode": self.cfg.run_mode,
            "frames_processed": len(todo),
            "elapsed_sec": round(time.time() - t0, 1)}
        Path(self.cfg.manifest_path).write_text(json.dumps(manifest, indent=2))
        print(f"\n✅ Done in {(time.time()-t0)/60:.1f} min")
        return manifest


agent = ContentExtractionAgent(CFG, model, processor)
manifest = agent.run(manifest)


In [ ]:
# @title 7. Review the extractions
frames = manifest["frames"]
by = {"model": 0, "heuristic": 0, "error": 0}
for f in frames:
    by[f.get("extraction_meta", {}).get("structured_by", "error")] = \
        by.get(f.get("extraction_meta", {}).get("structured_by", "error"), 0) + 1
print(f'{len(frames)} frames · structured by model: {by.get("model",0)} · '
      f'heuristic fallback: {by.get("heuristic",0)} · errors: {by.get("error",0)}\n')

for rec in frames[:3]:
    c = rec.get("extracted_content", {})
    print("═" * 70)
    print(f'{rec["frame_id"]} @ {rec["timestamp_hms"]}  '
          f'[{c.get("frame_type")}] (structured by {c.get("_structured_by")})')
    print("Title:", c.get("slide_title"))
    body = (c.get("content_text") or "")[:400]
    print("Content:", body + ("…" if len(body) == 400 else ""))
    if c.get("code"):
        print("Code:", c["code"][:200])


In [ ]:
# @title 8. Inspect one frame next to what the model extracted
import matplotlib.pyplot as plt

def show_frame(n: int = 0):
    rec = manifest["frames"][n]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(Image.open(rec["path"])); ax.axis("off")
    ax.set_title(f'{rec["frame_id"]} @ {rec["timestamp_hms"]}')
    plt.show()
    print(json.dumps(rec.get("extracted_content", {}), indent=2)[:2000])

show_frame(0)


In [ ]:
# @title 9. Package for Stage 4 (text-level duplicate elimination)
archive = Path(CFG.work_dir) / "stage3_output"
if archive.with_suffix(".zip").exists():
    archive.with_suffix(".zip").unlink()
staging = Path(CFG.work_dir) / "_staging"
if staging.exists(): shutil.rmtree(staging)
staging.mkdir()
shutil.copy(CFG.manifest_path, staging / "manifest.json")
zip_path = shutil.make_archive(str(archive), "zip", staging)
shutil.rmtree(staging)
print("Created:", zip_path)

# from google.colab import files; files.download(zip_path)
# shutil.copy(zip_path, "/content/drive/MyDrive/pipeline/stage3_output.zip")


## Manifest after this stage

Each kept frame now carries an `extracted_content` block:

```json
{"frame_id": "frame_0007", "timestamp_hms": "00:03:00", "status": "kept",
 "extracted_content": {
   "slide_title": "Vector embeddings",
   "content_text": "An embedding maps tokens to dense vectors…",
   "code": null,
   "diagram_description": "2-D scatter of word vectors, clusters labeled…",
   "frame_type": "slide"
 },
 "extraction_meta": {"model": "you/your-qwen", "mode": "local", "elapsed_sec": 8.2}}
```

**Stage 4** (duplicate eliminator) works purely on these `content_text` fields — no images needed from here on. Only `manifest.json` moves forward.